# Video RAG — Phase 1: Notebook 04
## Timestamp-Grounded Question Answering Pipeline

This notebook demonstrates the complete end-to-end question answering pipeline:
1. User asks a question about the video.
2. The query is embedded and matched against the FAISS index.
3. Top-k evidence chunks with timestamps are retrieved.
4. The LLM (Gemini API or grounded baseline) answers using **strictly** the retrieved context and cites timestamps.
5. Out-of-scope / unanswerable questions are cleanly refused.


In [ ]:
import sys
from pathlib import Path

sys.path.append(str(Path.cwd().parent))

from src.rag_pipeline import VideoRAGPipeline


### Step 1: Initialize the Video RAG Pipeline


In [ ]:
# Initialize pipeline with pre-built FAISS index
pipeline = VideoRAGPipeline()
pipeline.load_index("../data/vector_store")
pipeline.current_video = "sample.mp4"
print("Video RAG Pipeline ready.")


### Step 2: Test Grounded Question Answering


In [ ]:
def ask_and_display(question: str, top_k: int = 3):
    print("=" * 80)
    print(f"QUESTION: {question}")
    print("=" * 80)
    
    result = pipeline.query(question, top_k=top_k)
    
    print(f"\nANSWER:\n{result['answer']}\n")
    print(f"EVIDENCE CITATION: {result['evidence_raw']}")
    print(f"LATENCY: Retrieval={result['retrieval_latency_sec']}s | Generation={result['generation_latency_sec']}s | Total={result['total_latency_sec']}s")
    
    print("\nRETRIEVED CONTEXT CHUNKS:")
    for i, c in enumerate(result['retrieved_chunks'], 1):
        print(f"  [{i}] {c['chunk_id']} | Score: {c['score']:.4f} | Time: {c['formatted_time']}")
        print(f"      {c['text'][:100]}...")


In [ ]:
# Test Case 1: In-domain Definition Question
ask_and_display("What is the primary goal of database normalization?")


In [ ]:
# Test Case 2: Specific Concept Question
ask_and_display("What is required for a table to satisfy First Normal Form?")


In [ ]:
# Test Case 3: Negative Test Case (Out of Domain - Expected Refusal)
ask_and_display("What is the capital of France?")


### Step 3: Comparative Analysis — Baseline A vs Baseline B


In [ ]:
# Compare Baseline B (RAG Top-K Retrieval) with Baseline A (Full Transcript Context)
q = "How does Second Normal Form eliminate partial dependencies?"

print("--- Baseline B: Transcript RAG (Top-3) ---")
res_b = pipeline.query(q, top_k=3)
print(f"Answer: {res_b['answer']}")
print(f"Latency: {res_b['total_latency_sec']}s")


### Step 4: Optional Interactive Loop (Uncomment to use interactively)


In [ ]:
# Uncomment below to run an interactive terminal query loop:
# while True:
#     user_q = input("\nAsk a question about the video (or 'exit'): ")
#     if user_q.strip().lower() in ('exit', 'quit', 'q'):
#         break
#     ask_and_display(user_q)
